In [ ]:
pip install langchain langchain-google-genai pandas

In [ ]:
pip install langchain-classic

In [ ]:
import os
import sqlite3
import pandas as pd
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

In [ ]:
#Setup sqlite database and insert data

def setup_database():

  conn=sqlite3.connect("students.db")
  cursor=conn.cursor()

  cursor.execute("""
    CREATE TABLE IF NOT EXISTS students (
      student_id TEXT PRIMARY KEY,
      name TEXT,
      department TEXT,
      python INTEGER,
      database INTEGER,
      ai INTEGER,
      web INTEGER
  )
""")

  students_data=[
      ("22CS045", "Dhanushya", "Computer Science", 85, 72, 90, 78),
      ("22CS046", "Rahul", "Computer Science", 65, 70, 68, 72),
      ("22CS047", "Priya", "Information Technology", 92, 88, 95, 90),
      ("22CS048", "Arun", "Information Technology", 55, 60, 58, 62),
      ("22CS049", "Meena", "Computer Science", 78, 85, 80, 88),
  ]

  cursor.executemany("INSERT INTO students VALUES (?,?,?,?,?,?,?)", students_data)
  conn.commit()
  conn.close()

setup_database()

In [ ]:
#Define langchain tools

@tool
def get_student_info(student_id: str) -> dict:
    """Fetch the basic profile information (Name and Department) of a student using their Student ID."""
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("SELECT name, department FROM students WHERE student_id = ?", (student_id,))
    result = cursor.fetchone()
    conn.close()

    if result:
        return {"name": result[0], "department": result[1]}
    return {"error": f"Student with ID {student_id} not found."}

@tool
def get_student_marks(student_id: str) -> dict:
    """Fetch all subject marks (Python, Database, AI, Web) for a given student using their Student ID."""
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("SELECT python, database, ai, web FROM students WHERE student_id = ?", (student_id,))
    result = cursor.fetchone()
    conn.close()

    if result:
        return {
            "python": result[0],
            "database": result[1],
            "ai": result[2],
            "web": result[3]
        }
    return {"error": f"Marks for student ID {student_id} not found."}

@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression string to calculate total or average marks."""
    try:
        result = eval(expression, {"__builtins__": None}, {})
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"

@tool
def get_passing_rules() -> dict:
    """Retrieves the university passing rules and minimum mark thresholds."""
    return {
        "minimum_overall_average_percent": 40,
        "minimum_mark_per_subject": 35
    }


In [ ]:
os.environ["GOOGLE_API_KEY"]="AQ.Ab8RN6I7f8CWtFsEBpt0kwK_SGwT8JCq1qWNoyFYLq6baTkmAw"

In [ ]:
llm=ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

tools = [get_student_info, get_student_marks, calculator, get_passing_rules]

In [ ]:
agent_executor=create_react_agent(llm,tools)


In [ ]:
#Run the Agent

challenge_question = (
    "I am 22CS045. Tell me my name, department, total marks, average marks, "
    "and whether I satisfy the university passing requirements."
)

In [ ]:
final_message = agent_executor.invoke({"messages": [("user", challenge_question)]})

print("\n--- Final Answer ---")
print(final_message["messages"][-1].content)

if isinstance(final_message, list):
    clean_output = "".join([block["text"] for block in final_message if block.get("type") == "text"])
else:
    clean_output = final_message

print("\n--- Final Answer ---\n")
print(clean_output)